In [1]:
from datasets import load_dataset

# 加载 IMDB 数据集
dataset = load_dataset("imdb")
# print(dataset)
# DatasetDict({
#     train: Dataset({features: ['text', 'label'], num_rows: 25000})
#     test: Dataset({features: ['text', 'label'], num_rows: 25000})
# })

# 查看样本
# print(dataset["train"][0]["text"][:200])
# print(dataset["train"][0]["label"])  # 0=负面, 1=正面

# 各取 2000 条用于快速实验
# train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
# test_dataset = dataset["test"].shuffle(seed=42).select(range(500))
# print(f"训练集: {len(train_dataset)}, 测试集: {len(test_dataset)}")
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

In [2]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_fn(examples):
    """对文本进行 tokenize，截断到 256 tokens"""
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)


# 批量 tokenize
train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

# 设置 PyTorch 格式
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [3]:
from transformers import AutoModelForSequenceClassification

model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
from transformers import TrainingArguments

In [5]:
training_args = TrainingArguments(
    output_dir="./sentiment_model",  # 模型输出目录
    num_train_epochs=3,  # 训练轮数
    per_device_train_batch_size=16,  # 训练批大小
    per_device_eval_batch_size=32,  # 评估批大小
    learning_rate=2e-5,  # BERT 微调常用学习率
    weight_decay=0.01,  # L2 正则化
    eval_strategy="epoch",  # 每 epoch 评估一次
    save_strategy="epoch",  # 每 epoch 保存一次
    load_best_model_at_end=True,  # 训练结束加载最优模型
    metric_for_best_model="accuracy",  # 以 accuracy 选最优
    logging_steps=50,  # 每 50 步打印日志
    report_to="none",  # 不上报到 wandb 等平台
)

In [6]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

In [8]:
from transformers import Trainer

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.425900,0.288666,0.886000,0.889320
2,0.194600,0.325599,0.880000,0.885496
3,0.130900,0.338703,0.898000,0.898204


TrainOutput(global_step=375, training_loss=0.26493612480163575, metrics={'train_runtime': 118.5799, 'train_samples_per_second': 50.599, 'train_steps_per_second': 3.162, 'total_flos': 789333166080000.0, 'train_loss': 0.26493612480163575, 'epoch': 3.0})

In [11]:
trainer.save_model("./sentiment_model/final")
tokenizer.save_pretrained("./sentiment_model/final")

('./sentiment_model/final\\tokenizer_config.json',
 './sentiment_model/final\\special_tokens_map.json',
 './sentiment_model/final\\vocab.txt',
 './sentiment_model/final\\added_tokens.json',
 './sentiment_model/final\\tokenizer.json')

In [14]:
import numpy as np
from sklearn.metrics import classification_report
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics  # ✅必须传入！
)
results = trainer.evaluate(eval_dataset=test_dataset)
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1 Score: {results['eval_f1']:.4f}")

Accuracy: 0.8980
F1 Score: 0.8982


In [15]:
preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids

print(classification_report(labels, preds, target_names=["负面", "正面"]))

              precision    recall  f1-score   support

          负面       0.91      0.88      0.90       254
          正面       0.88      0.91      0.90       246

    accuracy                           0.90       500
   macro avg       0.90      0.90      0.90       500
weighted avg       0.90      0.90      0.90       500



In [16]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./sentiment_model/final",
    tokenizer="./sentiment_model/final"
)

Device set to use cuda:0


In [17]:
texts = [
    "This movie is absolutely wonderful, I loved every minute!",
    "Terrible film, waste of time and money.",
    "It was okay, nothing special but not bad either.",
]

In [18]:
results = classifier(texts)

In [19]:
for text, result in zip(texts, results):
    label = "正面" if result["label"] == "LABEL_1" else "负面"
    print(f"[{label}] {result['score']:.4f} | {text[:50]}...")

[正面] 0.9912 | This movie is absolutely wonderful, I loved every ...
[负面] 0.9845 | Terrible film, waste of time and money....
[负面] 0.8863 | It was okay, nothing special but not bad either....


In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "./sentiment_model/final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((76

In [21]:
text = "The acting was superb and the story kept me engaged."
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)

In [22]:
with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    pred = torch.argmax(probs, dim=-1).item()

In [23]:
label = "正面" if pred == 1 else "负面"
print(f"预测: {label}, 概率: {probs[0][pred]:.4f}")

预测: 正面, 概率: 0.9849
